In [ ]:
from astropy.io import fits
import clevar
import sys

sys.path.insert(0, "/global/homes/l/lettieri/gcrcatalogs-new/gcr-catalogs")
sys.path.insert(0, "/global/homes/l/lettieri/gcr-catalogs")
sys.path.insert(0, "../ame")

from clevar.match_metrics.recovery import ClCatalogFuncs as r_cf
from clevar.match_metrics.distances import ClCatalogFuncs as d_cf
from clevar.match_metrics.scaling import ClCatalogFuncs as s_cf
import clevar.match_metrics.scaling as scaling

import pandas as pd
from astropy.table import unique, Table
import astropy
from matplotlib import pyplot as plt
import numpy as np
from numcosmo_py import Ncm, Nc
from numcosmo_py.external.pyssc import pyssc as PySS
from numcosmo_py.helper import npa_to_seq
from numcosmo_py import sky_match
import time
Ncm.cfg_init()


import GCRCatalogs
print('GCRCatalogs =', GCRCatalogs.__version__, '|' ,'GCR =', GCRCatalogs.GCR.__version__)
from GCR import GCRQuery
%load_ext autoreload
%autoreload 2

In [ ]:
zmin = 0.0
zmax = 1.2
logMmin = 12.5
logMmax = 16.0
logRmin = 0
logRmax = 3 

# We are loading the cosmoDC2_redmapper catalalog with richness
redmapper_cat = GCRCatalogs.load_catalog("cosmoDC2_v1.1.4_redmapper_v0.8.1")
image_cat = GCRCatalogs.load_catalog("cosmoDC2_v1.1.4_image")

# Taking some important information about the fiducial cosmological model and survey region
redmapper_cosmology = redmapper_cat.cosmology
sky_area = redmapper_cat.sky_area

# Listing all quantities in the choosen catalog
#np.sort(redmapper_cat.list_all_quantities())
#print(np.sort(redmapper_cat.list_all_quantities()))



#image_cat['halo_mass'].size()

data_redmapper = redmapper_cat.get_quantities(
    ["cluster_id", "redshift", "redshift_err", "richness", "richness_err", "ra", "dec"],
    filters=[
        f"redshift > {zmin}",
        f"redshift < {zmax}",
        f"richness > {10**logRmin}",
        f"richness < {10**logRmax}",
    ],
)

data_dc2 = image_cat.get_quantities(
    ["halo_id" , "redshift_true", "halo_mass", "ra_true", "dec_true"],
    filters=[
        f"redshift_true > {zmin}",
        f"redshift_true < {zmax}",
        f"halo_mass > {10**logMmin}",
        f"halo_mass < {10**logMmax}",
        f"is_central == {True}", 
    ],
)

table_halos = Table(data_dc2)
table_halos['halo_mass'] = np.log10(table_halos['halo_mass'])
table_redmapper = Table(data_redmapper)

In [ ]:
area = 4109.3
cosmo = Nc.HICosmoDEXcdm()
cosmo.param_set_by_name("Omegax", redmapper_cosmology.Ode0)
cosmo.param_set_by_name("H0", redmapper_cosmology.H0.value)
cosmo.param_set_by_name("Omegab", redmapper_cosmology.Ob0)
cosmo.param_set_by_name("Omegac", redmapper_cosmology.Odm0)  # 0.2603
cosmo.param_set_by_name("w", -1.0)  # -1.0

prim = Nc.HIPrimPowerLaw.new()
prim.param_set_by_name("ln10e10ASA", 3.059917486993207)
prim.props.n_SA = redmapper_cosmology.n_s
reion = Nc.HIReionCamb.new()

cosmo.add_submodel(prim)
cosmo.add_submodel(reion)


reion = Nc.HIReionCamb.new()

cosmo.add_submodel(prim)
cosmo.add_submodel(reion)

dist = Nc.Distance.new(100.0)
dist.prepare(cosmo)

In [ ]:
table_redmapper = Table(data_redmapper)
min_mass = [12.5, 12.6, 12.7, 12.8, 12.9,  13, 13.3, 13.4, 13.5, 13.7, 13.8, 14]
mean_time_halo_m = []
mean_time_halo_b = []
mean_time_halo_c = []
mean_time_halo = []
std_time_halo_m  = []
std_time_halo_b  = []
std_time_halo_c  = []
std_time_halo = []
size_halo = []

halo_coordinates = {"RA":"ra_true" , "DEC":"dec_true" , "z":"redshift_true", "ID":"ids"}
halo_properties  = {"halo_mass":"mass","halo_id":"halo_id"}

detections_coordinates =  {"RA":"ra" , "DEC":"dec" , "z":"redshift"}
detections_properties  = {"richness":"R" , "redshift_err":"z_err" , "richness_err":"R_err","cluster_id":"cluster_id"}


for mass in min_mass:
    reduced_halos = table_halos[table_halos['halo_mass'] >= mass]
    size_halo.append(len(reduced_halos['halo_mass']))

    halos = sky_match.SkyMatch(query_data=reduced_halos, query_coordinates=halo_coordinates,match_data=table_redmapper,
                   match_coordinates=detections_coordinates)

    detections   = halos.invert_query_match()
    time_match_m = []
    time_match_b = []
    time_match_c = []
    time_match   = []

    for i in range(5):
        start = time.perf_counter()
        start_m = time.perf_counter()
        halos_matched =  halos.match_3d(cosmo, 100)
        end_m = time.perf_counter()        
        time_match_m.append(end_m - start_m) 

        
        mask_halos = halos_matched.filter_mask_by_distance(60)

        start_b = time.perf_counter()
        unique_halos = halos_matched.select_best(mask=mask_halos , selection_criteria=sky_match.SelectionCriteria.MORE_MASSIVE , more_massive_column='richness')
        end_b = time.perf_counter()       
        time_match_b.append(end_b - start_b)

        start_c = time.perf_counter()
        detections_matched =  detections.match_3d(cosmo, 100)
        mask_detections =  detections_matched.filter_mask_by_distance(60)

        unique_detections = detections_matched.select_best(mask=mask_detections , selection_criteria=sky_match.SelectionCriteria.MORE_MASSIVE , more_massive_column='halo_mass')

        cross_indices = unique_halos.get_cross_match_indices(unique_detections)
        
        end_c = time.perf_counter()
        time_match_c.append(end_c - start_c)

        end = time.perf_counter()
        time_match.append(end - start)

    time_match_m = np.array(time_match_m)
    time_match_b = np.array(time_match_b)
    time_match_c = np.array(time_match_c)
    time_match   = np.array(time_match)
    
    mean_time_halo_m.append(np.mean(time_match_m))
    std_time_halo_m.append(np.std(time_match_m))

    mean_time_halo_b.append(np.mean(time_match_b))
    std_time_halo_b.append(np.std(time_match_b))

    mean_time_halo_c.append(np.mean(time_match_c))
    std_time_halo_c.append(np.std(time_match_c))

    mean_time_halo.append(np.mean(time_match))
    std_time_halo.append(np.std(time_match))

mean_time_halo_m = np.array(mean_time_halo_m)
std_time_halo_m = np.array(std_time_halo_m)

mean_time_halo_b = np.array(mean_time_halo_b)
std_time_halo_b = np.array(std_time_halo_b)

mean_time_halo_c = np.array(mean_time_halo_c)
std_time_halo_c = np.array(std_time_halo_c)

In [ ]:
plt.errorbar(size_halo, mean_time_halo_m, yerr=std_time_halo_m, fmt='o',label="multiple matching")
#plt.xscale("log")
plt.legend()
plt.show()
plt.errorbar(size_halo, mean_time_halo_b, yerr=std_time_halo_b, fmt='o',label="best matching")
#plt.xscale("log")
plt.legend()
plt.show()
plt.errorbar(size_halo, mean_time_halo_c, yerr=std_time_halo_c, fmt='o',label="cross matching")
plt.legend()
#plt.xscale("log")
plt.show()
plt.errorbar(size_halo, mean_time_halo, yerr=std_time_halo, fmt='o',label="full matching")
plt.legend()
#plt.xscale("log")
plt.show()

In [ ]:
min_rich = [5, 7, 9, 10, 12, 14, 15, 17,  20, 22, 25, 30]
mean_time_cluster_m = []
mean_time_cluster_b = []
mean_time_cluster_c = []
mean_time_cluster = []
std_time_cluster_m  = []
std_time_cluster_b  = []
std_time_cluster_c  = []
std_time_cluster = []
size_cluster = []

# Using the same coordinate/property maps defined in your snippet
halo_coordinates = {"RA":"ra_true" , "DEC":"dec_true" , "z":"redshift_true", "ID":"ids"}
halo_properties  = {"halo_mass":"mass","halo_id":"halo_id"}

detections_coordinates = {"RA":"ra" , "DEC":"dec" , "z":"redshift"}
detections_properties  = {"richness":"R" , "redshift_err":"z_err" , "richness_err":"R_err","cluster_id":"cluster_id"}

# Assuming min_richness is your new list of thresholds
for richness_val in min_rich:
    # Filter the detection catalog (redmapper) by richness
    reduced_clusters = table_redmapper[table_redmapper['richness'] >= richness_val]
    size_cluster.append(len(reduced_clusters['richness']))

    # Initialize SkyMatch with Clusters as the Query Data
    clusters = sky_match.SkyMatch(query_data=reduced_clusters, 
                                  query_coordinates=detections_coordinates,
                                  match_data=table_halos,
                                  match_coordinates=halo_coordinates)

    halos = clusters.invert_query_match()
    time_match_m = []
    time_match_b = []
    time_match_c = []
    time_match   = []

    for i in range(5):
        start = time.perf_counter()
        
        # --- Part M: Initial Match ---
        start_m = time.perf_counter()
        clusters_matched = clusters.match_3d(cosmo, 100)
        end_m = time.perf_counter()        
        time_match_m.append(end_m - start_m) 

        # --- Part B: Selection (Best Halo for each Cluster) ---
        mask_clusters = clusters_matched.filter_mask_by_distance(60)

        start_b = time.perf_counter()
        # Note: We select the best halo based on mass for the given cluster
        unique_clusters = clusters_matched.select_best(mask=mask_clusters, 
                                                       selection_criteria=sky_match.SelectionCriteria.MORE_MASSIVE, 
                                                       more_massive_column='halo_mass')
        end_b = time.perf_counter()       
        time_match_b.append(end_b - start_b)

        # --- Part C: Cross-Match (Verification via Halos) ---
        start_c = time.perf_counter()
        halos_matched = halos.match_3d(cosmo, 100)
        mask_halos = halos_matched.filter_mask_by_distance(60)

        unique_halos = halos_matched.select_best(mask=mask_halos, 
                                                 selection_criteria=sky_match.SelectionCriteria.MORE_MASSIVE, 
                                                 more_massive_column='richness')

        cross_indices = unique_clusters.get_cross_match_indices(unique_halos)
        
        end_c = time.perf_counter()
        time_match_c.append(end_c - start_c)

        end = time.perf_counter()
        time_match.append(end - start)

    # Statistical Aggregation
    time_match_m = np.array(time_match_m)
    time_match_b = np.array(time_match_b)
    time_match_c = np.array(time_match_c)
    time_match   = np.array(time_match)
    
    mean_time_cluster_m.append(np.mean(time_match_m))
    std_time_cluster_m.append(np.std(time_match_m))

    mean_time_cluster_b.append(np.mean(time_match_b))
    std_time_cluster_b.append(np.std(time_match_b))

    mean_time_cluster_c.append(np.mean(time_match_c))
    std_time_cluster_c.append(np.std(time_match_c))

    mean_time_cluster.append(np.mean(time_match))
    std_time_cluster.append(np.std(time_match))

# Final Conversion to Numpy Arrays
mean_time_cluster_m = np.array(mean_time_cluster_m)
std_time_cluster_m = np.array(std_time_cluster_m)

mean_time_cluster_b = np.array(mean_time_cluster_b)
std_time_cluster_b = np.array(std_time_cluster_b)

mean_time_cluster_c = np.array(mean_time_cluster_c)
std_time_cluster_c = np.array(std_time_cluster_c)

In [ ]:
plt.errorbar(size_cluster, mean_time_cluster_m, yerr=std_time_halo_m, fmt='o', label="multiple matching")
plt.legend()
plt.xscale("log")
plt.show()
plt.errorbar(size_cluster, mean_time_cluster_b, yerr=std_time_halo_m, fmt='o', label="best matching")
plt.legend()
plt.xscale("log")
plt.show()
plt.errorbar(size_cluster, mean_time_cluster_c, yerr=std_time_halo_m, fmt='o', label="cross matching")
plt.legend()
plt.xscale("log")
plt.show()
plt.errorbar(size_cluster, mean_time_cluster, yerr=std_time_halo_m, fmt='o', label="full matching")
plt.legend()
plt.xscale("log")
plt.show()